In [ ]:
import os
import sqlite3
from typing import TypedDict, Annotated, List
import uuid

# --- Gradio for the UI ---
import gradio as gr
# --- LangChain & LangGraph Imports ---
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import InfoSQLDatabaseTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
# from langgraph.checkpoint.sqlite import SqliteSaver
from IPython.display import Image,display
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import Tool
from langchain_community.tools.file_management.write import WriteFileTool



In [ ]:
db_file = "company_final_app.db"
conn = sqlite3.connect(db_file, check_same_thread=False)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    budget INTEGER
)""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department_id INTEGER,
    salary INTEGER
)""")
cursor.execute("INSERT OR IGNORE INTO departments (id, name, budget) VALUES (1, 'Engineering', 500000)")
cursor.execute("INSERT OR IGNORE INTO departments (id, name, budget) VALUES (2, 'Sales', 300000)")
cursor.execute("INSERT OR IGNORE INTO employees (id, name, department_id, salary) VALUES (1, 'Alice', 1, 95000)")
cursor.execute("INSERT OR IGNORE INTO employees (id, name, department_id, salary) VALUES (2, 'Bob', 2, 80000)")
conn.commit()
conn.close()


In [ ]:
class State(TypedDict):
    messages: Annotated[list,add_messages]
    conversation_context: str  # Add this to maintain context


In [ ]:
db = SQLDatabase.from_uri(f"sqlite:///{db_file}")

In [ ]:
run_query=Tool(
    name="run_tool",
    func=db.run,
    description="Used to run SQL queries"
)

info_tool = Tool(
    name="info_tool",
    func=lambda table: db.get_table_info([table.strip()]),
    description="Use this tool to get column names and types of all tables in the database. Useful when the user asks about schema, columns, or structure of tables."
)


def list_tables(_: str = ""):
    return db.get_table_names()

list_table = Tool(
    name="list_tool",
    func=list_tables,
    description="Use this tool to get all table names in the database. Useful for questions like 'how many tables are there?', 'list all tables', or 'what tables exist?'"
)


In [ ]:
tools = [run_query, info_tool,list_table]

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash",google_api_key=os.getenv("GEMINI_API_KEY"))
llm_with_tools=llm.bind_tools(tools)

def runner_agent(state: State):
    """This agent's ONLY job is to use its tools to get data."""
    context = state.get("conversation_context", "")

    system_prompt = f"""You are a seasoned SQL professional. Your goal is to answer the user's question by using your tools.
Conversation Context: {context}

**IMPORTANT**: Your final response must be based *directly* on the information returned from your tools.
- Do not describe your own actions.
- Do not say "I ran the query".
- If a tool returns data, state that data clearly.
- If you are asked about queries indirectly, check in the conversation context for more info
- Consider the conversation context when interpreting questions like "name them", "show them", "list them"
- If you are asked about the database as a whole like "total tables" use the list tool

**Example:**
- User asks: "Who has the highest salary?"
- Tool returns: "[('Alice', 95000)]"
- Your final answer should be: "The employee with the highest salary is Alice."
**Example 2:**
- Previous context: User asked about department count, result was 2 departments
- User asks: "name them"
- You should understand they want the names of those 2 departments
"""

    # Use the simple tool-bound LLM. DO NOT use with_structured_output here.
    response = llm_with_tools.invoke([SystemMessage(content=system_prompt)] + state["messages"])
    current_question = state["messages"][-1].content if state["messages"] else ""
    updated_context = f"{context}\nUser asked: {current_question}"



    # Return the full AI message, which will include any tool calls
    return {
        "messages": [response],
        "conversation_context": updated_context
    }

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("runner_agent",runner_agent)
graph_builder.add_node("tool_node",ToolNode(tools=tools))

graph_builder.add_conditional_edges(
    "runner_agent",
    tools_condition,
    {
        # If the agent wants to call a tool, route to the 'tool_node'
        "tools": "tool_node",
        # Otherwise, the graph execution is finished
        END: END
    }
)
graph_builder.add_edge("tool_node","runner_agent")
graph_builder.set_entry_point("runner_agent")

memory = MemorySaver() # Switched to a persistent saver
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
SESSION_ID = str(uuid.uuid4())


async def chat(user_input: str, history: list):
    # Use a unique ID for each conversation session
    thread_id = str(hash(str(history)))
    config = {"configurable": {"thread_id": SESSION_ID}}

    # The initial input for the graph
    input_message = {
        "messages": [("user", user_input)],
    }

    # Call the graph and get the final state
    final_state = await graph.ainvoke(input_message, config=config)

    # The final answer is in the last message of the final state.
    final_answer = final_state["messages"][-1].content

    return final_answer

gr.ChatInterface(chat, title="Advanced SQL Database Assistant 🤖", description="Ask me questions about the company database!").launch()